In [14]:
import json
from pathlib import Path
import json
from datetime import datetime, timedelta

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import silhouette_score
from IPython.display import HTML


In [15]:
# DATA DIRECTORIES
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'newyorker_caption_contest'
JSON_DIR = DATA_DIR / 'contests.json'
CSV_DIR = DATA_DIR / 'data'
CSV_CLEAN_DIR = DATA_DIR / 'data_cleaned'
IMG_DIR = DATA_DIR / 'images'
DOC_THEMES_DIR = DATA_DIR / 'THEMES_RESULTS'/'docs_with_themes.csv'
THEMES_TFIDF_DIR=DATA_DIR/'THEMES_RESULTS'/'TF_IDF_themes_scores.csv'
THEMES_DIR=DATA_DIR/'THEMES_RESULTS'/'themes.csv'
Themes_aug_DIR = DATA_DIR / 'Themes_augmented.csv'
#flourish_dir=DATA_DIR/'flourish_theme_similarity.csv'

## Analyzing funiness over time

In [16]:
# Load themes
themes_df = pd.read_csv(THEMES_DIR)

# Sort by mean funniness and select top 10
top10_funny_themes = (
    themes_df
    .sort_values("mu_mean", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# Display / inspect
top10_funny_themes[
    ["theme_id", "theme_label", "mu_mean", "nb_instances"]
]


,theme_id,theme_label,mu_mean,nb_instances
0,2,Year,1.226651,21754
1,22,Know Feel,1.226381,21192
2,51,Evolution,1.225645,21676
3,58,Lawyer,1.225457,27958
4,39,Insurance Cover,1.225239,20720
5,63,Gentrification,1.224371,20270
6,20,Wear Mask,1.223832,21909
7,49,Tell,1.223448,37933
8,45,Marriage,1.223011,35378
9,44,Look Ant,1.222959,21734


I'm going to work with Olama generated themes

I added the Olama generated themes as a new column in the themes.csv called llm_theme_label

In [18]:

ollama_labels = (
    pd.read_csv(Themes_aug_DIR)
    .rename(columns={"caption_id": "theme_id", "gen_label": "llm_theme_label"})
)

# Keep original theme_label and add the llm_theme_label column
ollama_themes_df = themes_df.merge(ollama_labels, on="theme_id", how="left")
#save to csv
ollama_themes_df.to_csv(
    "themes_ollama.csv",
    index=False
)


In [19]:
ollama_themes_df.head(10)

,theme_id,theme_label,nb_instances,mu_rank,mu_mean,mu_precision,mu_not_funny,mu_somewhat_funny,mu_funny,TF_IDF_terms,llm_theme_label
0,0,Dog,28929,3221.119071,1.211799,0.047786,100.960863,27.160904,10.610151,"dog, pet, bark, leash, puppy, animal, vet, kno...",dog
1,1,Fall Asleep,18290,3005.333206,1.217083,0.054425,93.689978,27.131281,11.584067,"sleep, night, tired, dream, bed, wake, nap, as...",sleep
2,2,Year,21754,3161.643560,1.226651,0.051957,98.822745,30.887515,13.722948,"year, month, season, fall, date, time, age, th...",time
3,3,Work Home,35262,3162.187975,1.218767,0.055696,88.715088,25.710210,10.443817,"work, job, work home, home, office, say, think...",work
4,4,Clown,22401,3421.341284,1.205269,0.051263,88.639421,21.413864,8.350485,"clown, circus, laugh, funny, cop, joke, tell, ...",comedy
5,5,Elephant Room,17598,3431.416596,1.214021,0.054945,87.008303,23.395439,9.781778,"room, elephant, elephant room, door, talk, spa...",acknowledgment
6,6,Vegan,17207,3646.708316,1.198785,0.055109,84.196315,18.852676,6.405184,"vegan, gluten, gluten free, free, vegetarian, ...",dietary
7,7,Hell,16298,3134.965885,1.219931,0.056697,93.928703,26.945944,11.085716,"think, way, need, know, say, hell, look, thing...",understanding
8,8,Boy,66343,3121.448548,1.213524,0.053852,92.028956,24.131830,9.542740,"good, way, right, bad, thank, thing, guy, nice...",goodness
9,9,Bird,30228,3180.440007,1.191049,0.051456,94.127465,23.745897,9.198478,"bird, fly, wing, pigeon, duck, chicken, egg, f...",bird


Adding time data


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

docs_with_themes = pd.read_csv(DOC_THEMES_DIR)
llm_df = pd.read_csv(Themes_aug_DIR) 

# make sure contest_id is numeric
docs_with_themes["contest_id"] = docs_with_themes["contest_id"].astype(int)

base_publish = datetime.strptime("2016.02.22", "%Y.%m.%d")

# publish date from weekly schedule
docs_with_themes["contest_day_publish"] = docs_with_themes["contest_id"].apply(
    lambda cid: base_publish + timedelta(days=(cid - 510) * 7)
)

# year column
docs_with_themes["contest_publish_year"] = docs_with_themes["contest_day_publish"].dt.year

# quick sanity check
print(docs_with_themes[["contest_id", "contest_day_publish", "contest_publish_year"]]
      .drop_duplicates()
      .sort_values("contest_id")
      .head(10))


# Build mapping theme_id -> label
llm_map = dict(zip(llm_df["caption_id"].astype(int), llm_df["gen_label"]))

# Ensure theme_id is int
docs_with_themes["theme_id"] = docs_with_themes["theme_id"].astype(int)

# Add Ollama label column
docs_with_themes["theme_label_llm"] = docs_with_themes["theme_id"].map(llm_map).fillna("Unknown").str.title()

# Quick check
print(docs_with_themes[["theme_id", "theme_label_llm"]].drop_duplicates().head(10))

# save (optional)
#docs_with_themes.to_csv(DOC_THEMES_DIR, index=False)




       contest_id contest_day_publish  contest_publish_year
0             510          2016-02-22                  2016
3906          511          2016-02-29                  2016
7232          512          2016-03-07                  2016
11632         513          2016-03-14                  2016
15774         514          2016-03-21                  2016
19726         515          2016-03-28                  2016
23997         516          2016-04-04                  2016
29916         517          2016-04-11                  2016
37699         518          2016-04-18                  2016
42209         519          2016-04-25                  2016
   theme_id theme_label_llm
0        62       Transport
1        64           Trump
2        53        Commerce
3        76         Apparel
4        45        Marriage
5        79        Beverage
6        32      Psychology
7        30       Elevation
8        49   Communication
9        46        Footwear


In [42]:
docs_with_themes

,doc_id,contest_id,kind,text,clean_text,rank,mean,precision,votes,not_funny,somewhat_funny,funny,theme_id,theme_label,contest_day_publish,contest_year,contest_publish_year,theme_label_llm
0,image_510,510,image,A man is relaxing on a city street. Others are...,man relax city street go business step pass ma...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62,Train,2016-02-22,2016,2016,Transport
1,caption_510_0,510,caption,I'm a congressman--obstruction is my job.,congressman obstruction job,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0,64,Election,2016-02-22,2016,2016,Trump
2,caption_510_1,510,caption,"I'm what they mean when they say, 'The middle ...",mean middle class step,1.0,1.842105,0.191381,19.0,8.0,6.0,5.0,53,Market,2016-02-22,2016,2016,Commerce
3,caption_510_2,510,caption,Does this suit make me look flat?,suit look flat,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0,76,Clothing,2016-02-22,2016,2016,Apparel
4,caption_510_3,510,caption,"When the right woman comes along, I'll know it.",right woman come know,3.0,1.625000,0.116657,32.0,15.0,14.0,3.0,45,Marriage,2016-02-22,2016,2016,Marriage
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2263323,caption_895_4953,895,caption,Looks like we took a wrong turn at the Hubble ...,look take wrong turn hubble deep field,4953.0,1.023256,0.016347,86.0,84.0,2.0,0.0,26,Asteroid,2023-07-10,2023,2023,Astronomy
2263324,caption_895_4954,895,caption,Post-modern just doesn't do it for me.,post modern,4954.0,1.022222,0.015625,90.0,88.0,2.0,0.0,48,Painting,2023-07-10,2023,2023,Contemporary
2263325,caption_895_4955,895,caption,How was I supposed to know they covered themse...,suppose know cover find macy feel naked,4955.0,1.020833,0.014654,96.0,94.0,2.0,0.0,76,Clothing,2023-07-10,2023,2023,Apparel
2263326,caption_895_4956,895,caption,"No, they wanted us to teleport the giveaway ca...",want teleport giveaway car macy,4956.0,1.020619,0.020619,97.0,96.0,0.0,1.0,59,Vehicle,2023-07-10,2023,2023,Transportation


We aggregate caption-level funniness scores by year and theme to compute yearly mean funniness for each theme.

Because absolute funniness scores are very close to each other across years, we then compute a yearly baseline: the average funniness across all themes within each year. Subtracting this baseline from each theme’s yearly mean yields a funniness delta, which measures whether a theme is funnier (positive delta) or less funny (negative delta) than the typical theme in that same year. This normalization removes year-to-year global effects and allows meaningful comparisons over time.

Finally, since some visualization tools (e.g. Flourish bar chart races) require non-negative values, the deltas are shifted by subtracting the minimum delta observed across all years and themes, and adding a small constant. This preserves the relative differences between themes while ensuring all values are positive and suitable for visualization.

As a result, the funniness_shifted column captures relative, year-normalized theme funniness, enabling clear temporal comparisons without distortion from global trends.

In [56]:
yearly_theme_funniness = (
    docs_with_themes
    .groupby(["contest_publish_year", "theme_label_llm"])["mean"]
    .mean()
    .reset_index(name="mean_funniness")
)
# Compute yearly baseline funniness
yearly_theme_funniness["year_mean"] = (
    yearly_theme_funniness
    .groupby("contest_publish_year")["mean_funniness"]
    .transform("mean")
)
# delta from yearly mean
yearly_theme_funniness["funniness_delta"] = (
    yearly_theme_funniness["mean_funniness"]
    - yearly_theme_funniness["year_mean"]
)

# shift so everything is positive
min_delta = yearly_theme_funniness["funniness_delta"].min()
yearly_theme_funniness["funniness_shifted"] = (
    yearly_theme_funniness["funniness_delta"] - min_delta + 0.001
)

yearly_theme_funniness.head()


,contest_publish_year,theme_label_llm,mean_funniness,year_mean,funniness_delta,funniness_shifted
0,2016,Acknowledgment,1.191354,1.198834,-0.007480,0.056564
1,2016,Apparel,1.190021,1.198834,-0.008814,0.055230
2,2016,Astronomy,1.212468,1.198834,0.013633,0.077677
3,2016,Bakery,1.180602,1.198834,-0.018232,0.045811
4,2016,Baseball,1.191801,1.198834,-0.007033,0.057010


<we get the top 10 funniest themes>

In [57]:
global_theme_funniness = (
    docs_with_themes
    .groupby("theme_label_llm")["mean"]
    .mean()
    .reset_index(name="global_mean_funniness")
    .sort_values("global_mean_funniness", ascending=False)
)

top10_themes = global_theme_funniness.head(10)
top10_themes

,theme_label_llm,global_mean_funniness
58,Sensation,1.226381
24,Evolution,1.225645
38,Justice,1.225457
37,Insurance,1.225239
65,Time,1.224038
19,Covid-19,1.223832
41,Marriage,1.223011
45,Observe,1.222959
35,Housing,1.222467
13,Cognition,1.221831


We track them over time

In [58]:
top10_labels = top10_themes["theme_label_llm"]

top10_over_time = yearly_theme_funniness[
    yearly_theme_funniness["theme_label_llm"].isin(top10_labels)
]
#top10_over_time
top10_over_time

,contest_publish_year,theme_label_llm,mean_funniness,year_mean,funniness_delta,funniness_shifted
13,2016,Cognition,1.215333,1.198834,0.016499,0.080542
19,2016,Covid-19,1.195803,1.198834,-0.003032,0.061012
24,2016,Evolution,1.201042,1.198834,0.002207,0.066251
35,2016,Housing,1.190331,1.198834,-0.008504,0.055540
37,2016,Insurance,1.194944,1.198834,-0.003890,0.060153
...,...,...,...,...,...,...
577,2023,Justice,1.215454,1.193644,0.021810,0.085853
580,2023,Marriage,1.182138,1.193644,-0.011506,0.052538
584,2023,Observe,1.200326,1.193644,0.006682,0.070725
597,2023,Sensation,1.187951,1.193644,-0.005693,0.058350


Finally we build the csv file for the Flourish site

In [ ]:

flourish_funny_themes = (
    top10_over_time
    .pivot(
        index="theme_label_llm",
        columns="contest_publish_year",
        values="funniness_shifted"
    )
    .reset_index()
)
# Insert an empty column after the first column (theme labels)
flourish_funny_themes.insert(
    1,              # position (after column A)
    " ",            # column name (blank / spacer column)
    ""               
)
flourish_funny_themes.to_csv(
    "flourish_funny_themes_over_time_delta.csv",
    index=False
)


In [63]:
HTML("""
<div class="flourish-embed flourish-bar-chart-race" data-src="visualisation/26907938"><script src="https://public.flourish.studio/resources/embed.js"></script><noscript><img src="https://public.flourish.studio/visualisation/26907938/thumbnail" width="100%" alt="bar-chart-race visualization" /></noscript></div>
     """)